In [1]:
import torch
from torch import nn
from torch.nn import functional as F
import tiktoken

In [3]:
with open('./data/input.txt', 'r') as f:
    text = f.read()
len(text)

1115393

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
def encode(s):
    return [stoi[c] for c in s]
def decode(l):
    return ''.join([itos[i] for i in l])
print(encode("hello world"))
print(decode(encode("hello world")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]  
val_data = data[n:]

In [9]:
batch_size = 4
block_size = 8
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:', xb.shape)
print('targets:', yb.shape)

inputs: torch.Size([4, 8])
targets: torch.Size([4, 8])


In [15]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print('logits shape:', logits.shape)
print('loss:', loss)
temp = m(xb)
print(temp[0].shape)
print(temp[0][:, -1, :].shape)
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


logits shape: torch.Size([32, 65])
loss: tensor(4.6418, grad_fn=<NllLossBackward0>)
torch.Size([4, 8, 65])
torch.Size([4, 65])

k:w-A? Ge.EbWL &XPK$JmQcup&Gk
k-db?LKnwiQDeYccv&BnOZGP$ZVVoVze&P:utZxuct'!B
kEfTE.KUSrqrz&VFSP'guBUu


In [18]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(4000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [20]:
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))




game, HourJ!
ores the
Slpre h Fo ave hersal l IRDerds sbe ayordvady sis an t aSes moldithe tla s we merk acis,&doouf dousprrt houk, s fldqse In nkeaTqusealexeat ouce
We.
Caykwardiad wrdyou told I an!
s,y:; t mulars!
RKE: y tothes d g th s t ffe mavouplld,
RI t?
Th s sprd hin has, worme the hore O:
NGHr t and k spyGoue,JBeat YLISpinNeanKnodiono helenwino bB&zMokya:
TBUSTIsheashes th y, tooshe bremstresoP
Sw wind cober t CLucon cll!
Brth? pe.

BUFtelinthe r
STHISDUE. critunemypisthe IIAqVhathmmed


In [46]:
class ModelConfiguration:
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer, head_size=None):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.head_size = head_size if head_size is not None else n_embd // n_head
        self.n_embd = n_embd
        self.n_head = n_head
        self.n_layer = n_layer

In [47]:
model_config = ModelConfiguration(
    vocab_size = vocab_size,
    block_size = block_size,
    n_embd = 32,
    n_head = 4,
    n_layer = 4
)

In [69]:
class FFN(nn.Module):
    def __init__(self, model_config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(model_config.n_embd, 4 * model_config.n_embd),
            nn.ReLU(),
            nn.Linear(4 * model_config.n_embd, model_config.n_embd)
        )

    def forward(self, x):
        return self.net(x)

In [56]:
class Head(nn.Module):
    def __init__(self, model_config):
        super().__init__()
        self.model_config = model_config
        self.key = nn.Linear(model_config.n_embd,  model_config.head_size, bias=False)
        self.query = nn.Linear(model_config.n_embd,  model_config.head_size, bias=False)
        self.value = nn.Linear(model_config.n_embd,  model_config.head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(model_config.block_size, model_config.block_size)))
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * self.model_config.head_size**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        out = wei @ v
        #print('wei shape:', wei.shape, 'x shape:', x.shape, 'out.shape', out.shape)
        return out

In [79]:
class MultiHeadAttention(nn.Module):
    def __init__(self, model_config):
        super().__init__()
        self.heads = nn.ModuleList([Head(model_config) for _ in range(model_config.n_head)])
        self.proj = nn.Linear(model_config.n_embd, model_config.n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [80]:
class Block(nn.Module):
    def __init__(self, model_config):
        super().__init__()
        self.saheads = MultiHeadAttention(model_config)
        #add cross attention hereA
        self.ffn = FFN(model_config)
        self.ln1 = nn.LayerNorm(model_config.n_embd)
        self.ln2 = nn.LayerNorm(model_config.n_embd)

    def forward(self, x):
        x = x + self.saheads(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

In [81]:
class BigramLanguageModel2(nn.Module):
    def __init__(self, model_config):
        super().__init__()
        self.model_config = model_config
        self.token_embedding_table = nn.Embedding(model_config.vocab_size, model_config.n_embd)
        self.position_embedding_table = nn.Embedding(model_config.block_size, model_config.n_embd)
        self.blocks = nn.Sequential(*[Block(model_config) for _ in range(model_config.n_layer)], nn.LayerNorm(model_config.n_embd))
        self.lmHead = nn.Linear(model_config.n_embd, model_config.vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = token_emb + pos_emb
        x = self.blocks(x)
        logits = self.lmHead(x) 
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.model_config.block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    

In [82]:
m = BigramLanguageModel2(model_config)
logits, loss = m(xb, yb)
print('logits shape:', logits.shape)
print('loss:', loss)
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


logits shape: torch.Size([256, 65])
loss: tensor(4.3158, grad_fn=<NllLossBackward0>)

fO$glUO,BixukMlKsZMbT&'ZTQQ-Z'vpGI yyow$$scKOPfOlxgow-d$.F afX;-v,M'DIbTFnbyr?fhPbNWgdGI,EfYoNjj fgc


In [83]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [85]:
print(loss)
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


tensor(1.7545, grad_fn=<NllLossBackward0>)


WARWICK:
But wammen,'
And go,
When all shems.

DUKE OF Iy livze Anto, by tog of noble pose in fooul good, tybalt to he wistateiat;
Come waliefenend, not is to ploor sean:
Eliengn farget well, flower and server the camtom songe thou, who his encours,
Be should!
To note.

GRDclions and nunte
Messance and all resence untour trings?

PORCABINLET:
I shoped thy bothers?
He is be of Boch, and that be die.
Provelings,
What, survant ston,
Or veropputsel comfors; having your be and for too Vhapparing, he


In [ ]:
"""
tensor(2.0589, grad_fn=<NllLossBackward0>)

Go For be have crokely a prospatform
dom waink
Treast.'
But ith beats tiblow com wich, the hing the of for 'tean trut to suchly the st no 
somer the my wart the imseck fut ster;
Yeas to huck you hou to truor
O, hup in cyoubut thee a toring bul parvy.

And neill too comman:
Hod whorclam, geell hind,

"""